# E07 — Eşleştirilmiş Transformer İç Temsil Testi

Amaç: Discovery/Holdout ayrımı ile aday temsil boyutlarını random control boyutlarıyla karşılaştırmak.

**Not:** Bu notebook E07'de kullanılan yöntemi ve doğrulanmış sonuçları arşivler. E07'de henüz causal intervention uygulanmamıştır.

In [ ]:
# ============================================================
# E07 — EŞLEŞTİRİLMİŞ VERİ SETİ
# Amaç: Aynı cümle yapısını koruyarak dört concept grubunu
# Discovery ve Holdout olarak dengeli biçimde ayırmak.
# ============================================================

matched_groups = {
    'concept_A': ['The cat is on the table.','The cat is under the table.','The cat is near the table.','The cat is beside the table.','The cat is behind the table.','The cat is in front of the table.','The cat is above the table.','The cat is below the table.','The cat is inside the room.','The cat is outside the room.'],
    'concept_B': ['The dog is on the table.','The dog is under the table.','The dog is near the table.','The dog is beside the table.','The dog is behind the table.','The dog is in front of the table.','The dog is above the table.','The dog is below the table.','The dog is inside the room.','The dog is outside the room.'],
    'concept_C': ['The bird is on the table.','The bird is under the table.','The bird is near the table.','The bird is beside the table.','The bird is behind the table.','The bird is in front of the table.','The bird is above the table.','The bird is below the table.','The bird is inside the room.','The bird is outside the room.'],
    'concept_D': ['The book is on the table.','The book is under the table.','The book is near the table.','The book is beside the table.','The book is behind the table.','The book is in front of the table.','The book is above the table.','The book is below the table.','The book is inside the room.','The book is outside the room.']
}

discovery_groups = {k: v[:5] for k,v in matched_groups.items()}
holdout_groups = {k: v[5:] for k,v in matched_groups.items()}

assert sum(map(len, matched_groups.values())) == 40
assert sum(map(len, discovery_groups.values())) == 20
assert sum(map(len, holdout_groups.values())) == 20
print('Matched dataset doğrulaması tamamlandı.')

In [ ]:
# ============================================================
# E07 — TRANSFORMER VE CÜMLE TEMSİLİ
# Amaç: Her cümleyi 768 boyutlu iç temsil vektörüne çevirmek.
# Mean pooling: token temsillerinin ortalamasını alır.
# ============================================================

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'distilbert/distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
model.eval()
tokenizer.pad_token = tokenizer.eos_token

def get_sentence_representation(sentence):
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze(0)

test_representation = get_sentence_representation('The cat is on the table.')
print('Temsil boyutu:', test_representation.shape)

In [ ]:
# ============================================================
# E07 — DISCOVERY TEMSİLLERİ
# Amaç: Aday boyutları yalnızca Discovery verisinden seçmek.
# ============================================================

discovery_sentences = []
discovery_labels = []
for group_name, sentences in discovery_groups.items():
    discovery_sentences.extend(sentences)
    discovery_labels.extend([group_name] * len(sentences))

discovery_representations = torch.stack([get_sentence_representation(s) for s in discovery_sentences])

labels_array = np.array(discovery_labels)
group_means = {}
for group_name in matched_groups:
    idx = np.where(labels_array == group_name)[0]
    group_means[group_name] = discovery_representations[idx].mean(dim=0)

group_mean_matrix = torch.stack([group_means[g] for g in matched_groups])
dimension_separation = group_mean_matrix.max(dim=0).values - group_mean_matrix.min(dim=0).values
sorted_dimensions = torch.argsort(dimension_separation, descending=True)
candidate_dimensions = [int(x) for x in sorted_dimensions[:5]]
print('Aday boyutlar:', candidate_dimensions)

In [ ]:
# ============================================================
# E07 — RANDOM CONTROL
# Amaç: Adaylarla çakışmayan 20 rastgele boyutu kontrol grubu yapmak.
# ============================================================

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
available = [i for i in range(768) if i not in candidate_dimensions]
random_control_dimensions = rng.choice(available, size=20, replace=False).tolist()

assert not set(candidate_dimensions).intersection(random_control_dimensions)
print('Random control:', random_control_dimensions)

In [ ]:
# ============================================================
# E07 — L1 ÖLÇÜMÜ
# Amaç: Bir boyutun concept grupları arasındaki ortalama mutlak
# farkını (L1) ölçmek. Bu metrik temsil ayrışmasıdır; nedensellik değildir.
# ============================================================

def calculate_dimension_l1(representations, labels, group_names, dimension):
    values_by_group = {}
    for group_name in group_names:
        idx = [i for i, label in enumerate(labels) if label == group_name]
        values_by_group[group_name] = representations[idx, dimension].cpu().numpy()
    distances = []
    for i in range(len(group_names)):
        for j in range(i + 1, len(group_names)):
            distances.extend(np.abs(values_by_group[group_names[i]] - values_by_group[group_names[j]]).tolist())
    return float(np.mean(distances))

candidate_l1_discovery = {d: calculate_dimension_l1(discovery_representations, discovery_labels, list(matched_groups), d) for d in candidate_dimensions}
control_l1_discovery = {d: calculate_dimension_l1(discovery_representations, discovery_labels, list(matched_groups), d) for d in random_control_dimensions}

print('Discovery aday ortalama L1:', np.mean(list(candidate_l1_discovery.values())))
print('Discovery control ortalama L1:', np.mean(list(control_l1_discovery.values())))

In [ ]:
# ============================================================
# E07 — HOLDOUT DOĞRULAMASI VE SIRALAMA
# Amaç: Discovery'de seçilen adayların bağımsız Holdout verisinde
# random control boyutlarından üstün kalıp kalmadığını ölçmek.
# ============================================================

holdout_sentences = []
holdout_labels = []
for group_name, sentences in holdout_groups.items():
    holdout_sentences.extend(sentences)
    holdout_labels.extend([group_name] * len(sentences))

holdout_representations = torch.stack([get_sentence_representation(s) for s in holdout_sentences])

candidate_l1_holdout = {d: calculate_dimension_l1(holdout_representations, holdout_labels, list(matched_groups), d) for d in candidate_dimensions}
control_l1_holdout = {d: calculate_dimension_l1(holdout_representations, holdout_labels, list(matched_groups), d) for d in random_control_dimensions}

all_discovery = {**candidate_l1_discovery, **control_l1_discovery}
all_holdout = {**candidate_l1_holdout, **control_l1_holdout}
discovery_rank = {d: r for r,d in enumerate(sorted(all_discovery, key=all_discovery.get, reverse=True), 1)}
holdout_rank = {d: r for r,d in enumerate(sorted(all_holdout, key=all_holdout.get, reverse=True), 1)}

print('Boyut | Discovery sıra | Holdout sıra')
for d in candidate_dimensions:
    print(d, discovery_rank[d], holdout_rank[d])

candidate_mean_discovery = np.mean(list(candidate_l1_discovery.values()))
candidate_mean_holdout = np.mean(list(candidate_l1_holdout.values()))
control_mean_discovery = np.mean(list(control_l1_discovery.values()))
control_mean_holdout = np.mean(list(control_l1_holdout.values()))

print('Discovery aday/control:', candidate_mean_discovery / control_mean_discovery)
print('Holdout aday/control:', candidate_mean_holdout / control_mean_holdout)
print('Holdout aday ortalamasını geçen control:', sum(v > candidate_mean_holdout for v in control_l1_holdout.values()), '/ 20')

## E07 doğrulanmış sonuç

- Aday boyutlar: `[430, 496, 36, 374, 314]`
- Discovery ortalama L1: **2.127904**
- Discovery control ortalama L1: **0.123722**
- Discovery oranı: **17.20×**
- Holdout ortalama L1: **1.962849**
- Holdout control ortalama L1: **0.124925**
- Holdout oranı: **15.71×**
- Holdout'ta adayı geçen control: **0/20**
- Adayların Discovery sırası: `496, 430, 314, 36, 374`
- Adayların Holdout sırası: `496, 430, 36, 314, 374`

**Yorum:** Adaylar bağımsız Holdout verisinde de güçlü L1 ayrışmasını korudu. Bu sonuç temsil düzeyinde adaylığı destekler; formal istatistiksel anlamlılık ve nedensel intervention E09/E08 kapsamında ayrıca test edilmelidir.